## Data import from Snowflake

In [0]:
customers = spark.table('retail_snowflake_connection_catalog.public.customers')
inventory = spark.table('retail_snowflake_connection_catalog.public.inventory')
promotions = spark.table('retail_snowflake_connection_catalog.public.promotions')
sales = spark.table('retail_snowflake_connection_catalog.public.sales')
skus = spark.table('retail_snowflake_connection_catalog.public.skus')
stores = spark.table('retail_snowflake_connection_catalog.public.stores')

In [0]:
display(sales.limit(10))

DATE,STORE_ID,SKU_ID,CUSTOMER_ID,QUANTITY,UNIT_PRICE,TOTAL_VALUE,CHANNEL,DISCOUNT_PCT
2021-01-01,41,1009,751.0,4,90.95,363.80,Store,0.0
2021-01-01,24,1038,null,1,84.39,84.39,Store,15.0
2021-01-01,47,1164,null,3,166.80,500.40,Store,15.0
2021-01-01,6,1046,2651.0,2,49.46,98.92,Noon,15.0
2021-01-01,29,1117,null,1,9.94,9.94,MobileApp,0.0
2021-01-01,5,1199,2234.0,3,195.41,586.23,Store,15.0
2021-01-01,21,1027,null,1,5.14,5.14,Website,15.0
2021-01-01,20,1098,1125.0,5,88.00,440.00,Amazon.ae,15.0
2021-01-01,1,1195,1982.0,2,17.78,35.56,Store,15.0
2021-01-01,37,1155,null,3,19.45,58.35,Store,15.0


## Checking row counts

In [0]:
print('customers: ', customers.count())
print('inventory: ', inventory.count())
print('promotions: ', promotions.count())
print('sales: ', sales.count())
print('skus: ', skus.count())
print('stores: ', stores.count())

customers:  5000
inventory:  8735
promotions:  33
sales:  641798
skus:  200
stores:  50


## Data Conversion to Pandas

In [0]:
import pandas as pd

In [0]:
customers_pd = customers.toPandas()
inventory_pd = inventory.toPandas()
promotions_pd = promotions.toPandas()
sales_pd = sales.toPandas()
skus_pd = skus.toPandas()
stores_pd = stores.toPandas()

In [0]:
print("Customers:", customers_pd.shape)
print("Inventory:", inventory_pd.shape)
print("Promotions:", promotions_pd.shape)
print("Sales:", sales_pd.shape)
print("SKUs:", skus_pd.shape)
print("Stores:", stores_pd.shape)

Customers: (5000, 7)
Inventory: (8735, 7)
Promotions: (33, 6)
Sales: (641798, 9)
SKUs: (200, 7)
Stores: (50, 5)


## Dataframe Preparation

In [0]:
print(sales_pd.columns)
print(skus_pd.columns)
print(stores_pd.columns)

Index(['DATE', 'STORE_ID', 'SKU_ID', 'CUSTOMER_ID', 'QUANTITY', 'UNIT_PRICE',
       'TOTAL_VALUE', 'CHANNEL', 'DISCOUNT_PCT'],
      dtype='object')
Index(['SKU_ID', 'SKU_NAME', 'CATEGORY', 'SUBCATEGORY', 'UNIT_PRICE',
       'COST_PRICE', 'BRAND'],
      dtype='object')
Index(['STORE_ID', 'STORE_NAME', 'CITY', 'STORE_TYPE', 'OPENING_DATE'], dtype='object')


In [0]:
sales_pd.columns = sales_pd.columns.str.lower()
customers_pd.columns = customers_pd.columns.str.lower()
inventory_pd.columns = inventory_pd.columns.str.lower()
promotions_pd.columns = promotions_pd.columns.str.lower()
skus_pd.columns = skus_pd.columns.str.lower()
stores_pd.columns = stores_pd.columns.str.lower()

In [0]:
sales_analysis = sales_pd.merge(
    skus_pd,on='sku_id',how='left')

sales_analysis = sales_analysis.merge(
    stores_pd,on='store_id',how='left')

In [0]:
print("Sales rows:", sales_pd.shape[0])
print("Sales analysis rows:", sales_analysis.shape[0])
print("Sales analysis columns:", sales_analysis.shape[1])

Sales rows: 641798
Sales analysis rows: 641798
Sales analysis columns: 19


## Descriptive Analysis - Key Sales KPIs

In [0]:
total_sales = sales_analysis['total_value'].sum()
units_sold = sales_analysis['quantity'].sum()
sales_records = len(sales_analysis)
avg_sales_value = sales_analysis['total_value'].mean()

print('Total Sales:', round(total_sales,2))
print('Units Sold:', units_sold)
print('Sales Records:', sales_records)
print('Average Sales Value:', round(avg_sales_value,2))


Total Sales: 73209782.99
Units Sold: 1677123
Sales Records: 641798
Average Sales Value: 114.07


## Sales Trend Analysis

In [0]:
sales_analysis['year'] = pd.to_datetime(sales_analysis['date']).dt.year

In [0]:
yearly_sales = sales_analysis.groupby('year')['total_value'].sum().reset_index()

yearly_sales['total_value'] = yearly_sales['total_value'].round(2)

yearly_sales

,year,total_value
0,2021,15328941.50
1,2022,14900959.25
2,2023,14575050.35
3,2024,15340701.22
4,2025,13064130.67


Observation: Annual sales remained relatively stable from 2021–2024, ranging between approximately 14.6M and 15.3M. Sales in 2025 are 13.06M through October, so they are not directly comparable with the complete previous years.

## Monthly Sales Trend Analysis

In [0]:
sales_analysis['year_month'] = pd.to_datetime(
    sales_analysis['date']
).dt.to_period('M')

In [0]:
monthly_sales = (
    sales_analysis.groupby('year_month')['total_value'].sum().reset_index()
)

monthly_sales['total_value'] = monthly_sales['total_value'].round(2)

monthly_sales

,year_month,total_value
0,2021-01,1980260.06
1,2021-02,1985618.84
2,2021-03,830610.94
3,2021-04,1205290.58
4,2021-05,1714872.08
5,2021-06,1102448.76
6,2021-07,1593500.17
7,2021-08,1534914.44
8,2021-09,743017.49
9,2021-10,746654.89


Observation: Monthly sales show a recurring seasonal pattern. Higher sales are generally observed during January–February and July–August, while September–October tend to record lower sales.

## Category-wise Sales Analysis

In [0]:
category_sales = (
    sales_analysis.groupby('category')['total_value'].sum().sort_values(ascending=False).reset_index()
)

category_sales['total_value'] = category_sales['total_value'].round(2)

category_sales

,category,total_value
0,Electronics,40790950.89
1,Household,8190735.92
2,Personal Care,8181799.29
3,Grocery,5564702.41
4,Snacks,4321275.25
5,Dairy,4238161.64
6,Beverages,1922157.59


Observation: Electronics is the highest revenue-generating category with sales of approximately 40.79M, contributing substantially more than any other category. Household and Personal Care are the next highest-performing categories.

## Store-Performance Analysis

In [0]:
store_sales = (
    sales_analysis.groupby('store_name')['total_value'].sum().sort_values(ascending=False).reset_index()
)

store_sales['total_value'] = store_sales['total_value'].round(2)

store_sales.head(10)

,store_name,total_value
0,BlueMart Store 41,1500094.91
1,BlueMart Store 11,1495245.74
2,BlueMart Store 49,1494703.15
3,BlueMart Store 44,1493356.50
4,BlueMart Store 04,1492536.75
5,BlueMart Store 37,1489346.54
6,BlueMart Store 35,1486546.13
7,BlueMart Store 08,1486405.93
8,BlueMart Store 17,1484028.19
9,BlueMart Store 19,1481028.59


Observation: Store performance is relatively balanced among the top-performing stores. 

## Product-Performance Analysis

In [0]:
product_sales = (
    sales_analysis.groupby('sku_name')['total_value'].sum().sort_values(ascending=False).reset_index()
)

product_sales['total_value'] = product_sales['total_value'].round(2)

product_sales.head(10)

,sku_name,total_value
0,Electronics_Batteries_1035,1968289.04
1,Electronics_Chargers_1072,1882741.67
2,Electronics_Mobile Accessories_1174,1851457.18
3,Electronics_Cables_1199,1788582.29
4,Electronics_Cables_1019,1592031.54
5,Electronics_Cables_1164,1541797.14
6,Electronics_Cables_1037,1427421.10
7,Electronics_Batteries_1159,1419403.95
8,Electronics_Cables_1078,1403360.74
9,Electronics_Chargers_1065,1383637.85


Observation: The Top 10 products by sales all belong to the Electronics category, supporting Electronics as the dominant revenue-generating category. 

## Sales-Channel Analysis

In [0]:
channel_sales = (
    sales_analysis.groupby('channel')['total_value'].sum().sort_values(ascending=False).reset_index()
)

channel_sales['total_value'] = channel_sales['total_value'].round(2)

channel_sales

,channel,total_value
0,Store,30227603.54
1,Website,20299836.60
2,MobileApp,13097435.70
3,Amazon.ae,6470717.92
4,Noon,3114189.23


## Customer-Segment Analysis

In [0]:
customer_analysis = sales_pd.merge(customers_pd,left_on='customer_id',right_on='cust_id',how='left'
)

In [0]:
customer_analysis.shape

(641798, 16)

In [0]:
segment_sales = (
    customer_analysis.groupby('loyalty_segment')['total_value'].sum().sort_values(ascending=False)
    .reset_index()
)

segment_sales['total_value'] = segment_sales['total_value'].round(2)

segment_sales

,loyalty_segment,total_value
0,Silver,31915246.61
1,Gold,17059504.31
2,Platinum,5902103.35


## Identified vs Unidentified Customer Sales

In [0]:
identified_sales = customer_analysis[customer_analysis['customer_id'].notna()]['total_value'].sum()

unidentified_sales = customer_analysis[customer_analysis['customer_id'].isna()]['total_value'].sum()

print("Identified Customer Sales:", round(identified_sales, 2))
print("Unidentified Customer Sales:", round(unidentified_sales, 2))

Identified Customer Sales: 54876854.27
Unidentified Customer Sales: 18332928.72


## Promotion & Discount Analysis

In [0]:
sales_analysis['discount_status'] = sales_analysis['discount_pct'].apply(
    lambda x: 'Discounted' if x > 0 else 'Non-Discounted'
)

In [0]:
discount_analysis = (sales_analysis.groupby('discount_status')
    .agg(
        total_sales=('total_value', 'sum'),
        sales_records=('total_value', 'count')
    ).reset_index()
)

discount_analysis['total_sales'] = discount_analysis['total_sales'].round(2)

discount_analysis

,discount_status,total_sales,sales_records
0,Discounted,24905888.10,204357
1,Non-Discounted,48303894.89,437441


## Promotion Analysis

In [0]:
promotion_summary = (promotions_pd.groupby('promo_type')
    .agg(
        number_of_promotions=('promo_id', 'count'),
        avg_discount=('discount_pct', 'mean')
    ).reset_index()
)

promotion_summary['avg_discount'] = promotion_summary['avg_discount'].round(2)

promotion_summary

,promo_type,number_of_promotions,avg_discount
0,Black Friday,4,33.75
1,DSF,5,17.0
2,Eid,10,24.5
3,National Day,4,15.0
4,Ramadan,5,23.0
5,Summer Sale,5,12.0


## Inventory Analysis

In [0]:
inventory_pd['stock_status'] = inventory_pd.apply(
    lambda x: 'Reorder Required'
    if x['stock_on_hand'] <= x['reorder_point']
    else 'Sufficient Stock',
    axis=1
)

In [0]:
inventory_summary = (
    inventory_pd.groupby('stock_status').size()
    .reset_index(name='sku_store_records')
)

inventory_summary

,stock_status,sku_store_records
0,Sufficient Stock,8735


# Predictive Modelling 
## Product-Demand Prediction
### Target Variable - quantity

### Data Preparation

In [0]:
model_data = sales_analysis[
    ['date', 'store_id', 'sku_id', 'channel', 'discount_pct', 'quantity']
].copy()

model_data.head()

,date,store_id,sku_id,channel,discount_pct,quantity
0,2021-01-01,41,1009,Store,0.0,4
1,2021-01-01,24,1038,Store,15.0,1
2,2021-01-01,47,1164,Store,15.0,3
3,2021-01-01,6,1046,Noon,15.0,2
4,2021-01-01,29,1117,MobileApp,0.0,1


### Date Extractions

In [0]:
model_data['date'] = pd.to_datetime(model_data['date'])

model_data['year'] = model_data['date'].dt.year
model_data['month'] = model_data['date'].dt.month
model_data['day_of_week'] = model_data['date'].dt.dayofweek

model_data.head()

,date,store_id,sku_id,channel,discount_pct,quantity,year,month,day_of_week
0,2021-01-01,41,1009,Store,0.0,4,2021,1,4
1,2021-01-01,24,1038,Store,15.0,1,2021,1,4
2,2021-01-01,47,1164,Store,15.0,3,2021,1,4
3,2021-01-01,6,1046,Noon,15.0,2,2021,1,4
4,2021-01-01,29,1117,MobileApp,0.0,1,2021,1,4


In [0]:
model_data = model_data.drop('date', axis=1)

### Input & Target

In [0]:
X = model_data.drop('quantity', axis=1)
y = model_data['quantity']

In [0]:
X.head()

,store_id,sku_id,channel,discount_pct,year,month,day_of_week
0,41,1009,Store,0.0,2021,1,4
1,24,1038,Store,15.0,2021,1,4
2,47,1164,Store,15.0,2021,1,4
3,6,1046,Noon,15.0,2021,1,4
4,29,1117,MobileApp,0.0,2021,1,4


In [0]:
y.head()

0    4
1    1
2    3
3    2
4    1
Name: quantity, dtype: object

### Categorical columns

In [0]:
X = pd.get_dummies(
    X,
    columns=['store_id', 'sku_id', 'channel'],
    drop_first=True
)

X.head()

,discount_pct,year,month,day_of_week,store_id_2,store_id_3,store_id_4,store_id_5,store_id_6,store_id_7,store_id_8,store_id_9,store_id_10,store_id_11,store_id_12,store_id_13,store_id_14,store_id_15,store_id_16,store_id_17,store_id_18,store_id_19,store_id_20,store_id_21,store_id_22,store_id_23,store_id_24,store_id_25,store_id_26,store_id_27,store_id_28,store_id_29,store_id_30,store_id_31,store_id_32,store_id_33,store_id_34,store_id_35,store_id_36,store_id_37,...,sku_id_1165,sku_id_1166,sku_id_1167,sku_id_1168,sku_id_1169,sku_id_1170,sku_id_1171,sku_id_1172,sku_id_1173,sku_id_1174,sku_id_1175,sku_id_1176,sku_id_1177,sku_id_1178,sku_id_1179,sku_id_1180,sku_id_1181,sku_id_1182,sku_id_1183,sku_id_1184,sku_id_1185,sku_id_1186,sku_id_1187,sku_id_1188,sku_id_1189,sku_id_1190,sku_id_1191,sku_id_1192,sku_id_1193,sku_id_1194,sku_id_1195,sku_id_1196,sku_id_1197,sku_id_1198,sku_id_1199,sku_id_1200,channel_MobileApp,channel_Noon,channel_Store,channel_Website
0,0.0,2021,1,4,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False
1,15.0,2021,1,4,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False
2,15.0,2021,1,4,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False
3,15.0,2021,1,4,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False
4,0.0,2021,1,4,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False


In [0]:
X.shape

(641798, 256)

### Train-Test Split

In [0]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

Training rows: 513438
Testing rows: 128360


### Model Training - Linear Regression

In [0]:
from sklearn.linear_model import LinearRegression

lr_model = LinearRegression()

lr_model.fit(X_train, y_train)

print("Linear Regression training completed.")

Linear Regression training completed.


### Model Evaluation

In [0]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

y_pred = lr_model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("MAE:", round(mae, 3))
print("RMSE:", round(rmse, 3))
print("R² Score:", round(r2, 3))

MAE: 1.193
RMSE: 1.432
R² Score: 0.065


### Model Training - Randon Forest Regression

In [0]:
from sklearn.ensemble import RandomForestRegressor

rf_model = RandomForestRegressor(
    n_estimators=50,
    max_depth=10,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

print("Random Forest training completed.")

Random Forest training completed.


In [0]:
rf_pred = rf_model.predict(X_test)

rf_mae = mean_absolute_error(y_test, rf_pred)
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_pred))
rf_r2 = r2_score(y_test, rf_pred)

print("Random Forest MAE:", round(rf_mae, 3))
print("Random Forest RMSE:", round(rf_rmse, 3))
print("Random Forest R² Score:", round(rf_r2, 3))

Random Forest MAE: 1.127
Random Forest RMSE: 1.38
Random Forest R² Score: 0.131


In [0]:
results = pd.DataFrame({
    'Actual Quantity': y_test.values,
    'Predicted Quantity': rf_pred
})

results.head(10)

,Actual Quantity,Predicted Quantity
0,4,2.460715
1,1,2.037588
2,2,2.011266
3,1,1.997329
4,1,2.068184
5,1,3.144287
6,3,3.155517
7,2,2.013640
8,1,2.037142
9,3,3.154507
